In [5]:
#!/usr/bin/env python3
"""
Improved: visualize_one_class_sets_auto_fixed.py

- Auto-detects dataset layout (train/images + train/labels OR images+txt in train)
- Defaults to --data-root ./train so you can just run: python visualize_one_class_sets_auto_fixed.py
- Prints helpful errors instead of SystemExit:2
- Emits full tracebacks on unexpected exceptions to help debugging

Usage (example):
  # run from parent folder that contains 'train'
  python visualize_one_class_sets_auto_fixed.py

  # or explicitly:
  python visualize_one_class_sets_auto_fixed.py --data-root ./train --output-dir ./class_sets --names-file ./data5.yaml
"""
import argparse
from pathlib import Path
import random
import cv2
import yaml
import sys
import traceback

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

DEFAULT_NAMES = {
    0: "general_agents",
    1: "wheeled_assist",
    2: "non_wheeled_assist",
    3: "load_objects",
    4: "personal_transport",
}

def load_names_from_yaml(path: Path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            doc = yaml.safe_load(f)
        names = doc.get("names")
        if isinstance(names, dict):
            return {int(k): v for k, v in names.items()}
        if isinstance(names, list):
            return {i: n for i, n in enumerate(names)}
    except Exception as e:
        print(f"Warning: couldn't load names from {path}: {e}")
    return DEFAULT_NAMES

def gather_images_and_labels(data_root: Path):
    data_root = data_root.resolve()
    if not data_root.exists():
        raise FileNotFoundError(f"data root does not exist: {data_root}")

    # Try common layout: data_root/images and data_root/labels
    images_dir = None
    labels_dir = None

    cand_images_dirs = [data_root / "images", data_root]
    for d in cand_images_dirs:
        if d.exists():
            imgs = [p for p in d.rglob("*") if p.suffix.lower() in IMAGE_EXTS]
            if imgs:
                images_dir = d
                break

    cand_labels_dirs = [data_root / "labels", data_root]
    for d in cand_labels_dirs:
        if d.exists():
            txts = [p for p in d.rglob("*.txt")]
            if txts:
                labels_dir = d
                break

    if images_dir and not labels_dir:
        all_txts = list(data_root.rglob("*.txt"))
        if all_txts:
            parent_counts = {}
            for t in all_txts:
                parent = t.parent
                parent_counts[parent] = parent_counts.get(parent, 0) + 1
            labels_dir = max(parent_counts, key=parent_counts.get)

    if not images_dir:
        raise FileNotFoundError(f"No images found in {data_root} or {data_root/'images'}")

    if not labels_dir:
        labels_dir = images_dir  # co-located fallback

    images = sorted([p for p in images_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS])
    return images, labels_dir

def read_yolo_labels(label_path):
    if not label_path.exists():
        return []
    boxes = []
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                cls = int(float(parts[0]))
                x_c, y_c, w, h = map(float, parts[1:5])
                boxes.append((cls, x_c, y_c, w, h))
            except Exception:
                continue
    return boxes

def yolo_to_xyxy(box, img_w, img_h):
    cls, x_c, y_c, w, h = box
    x1 = int((x_c - w/2.0)*img_w)
    y1 = int((y_c - h/2.0)*img_h)
    x2 = int((x_c + w/2.0)*img_w)
    y2 = int((y_c + h/2.0)*img_h)
    x1 = max(0, min(x1, img_w-1))
    y1 = max(0, min(y1, img_h-1))
    x2 = max(0, min(x2, img_w-1))
    y2 = max(0, min(y2, img_h-1))
    return cls, x1, y1, x2, y2

def draw_boxes(image, boxes_xyxy, class_name, thickness=2):
    for _, x1, y1, x2, y2 in boxes_xyxy:
        cv2.rectangle(image, (x1, y1), (x2, y2), (0,255,0), thickness)
    cv2.putText(image, class_name, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2, cv2.LINE_AA)
    return image

def augment_image(img, mode):
    if mode == 0:
        return img.copy()
    if mode == 1:
        return cv2.flip(img, 1)
    if mode == 2:
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        h,s,v = cv2.split(hsv)
        v = cv2.add(v, 30)
        v = cv2.min(v, 255)
        hsv2 = cv2.merge((h,s,v))
        return cv2.cvtColor(hsv2, cv2.COLOR_HSV2BGR)
    return img.copy()

def run_visualization(data_root_str, output_dir_str, names_file, seed):
    data_root = Path(data_root_str)
    images, labels_dir = gather_images_and_labels(data_root)
    print(f"Images found: {len(images)} (from {images[0].parent if images else 'n/a'})")
    print(f"Labels folder: {labels_dir}")

    names = DEFAULT_NAMES
    if names_file:
        names = load_names_from_yaml(Path(names_file))
    class_ids = sorted(names.keys())
    class_names = {cid: names[cid] for cid in class_ids}
    print("Classes:", class_names)

    image_to_labels = {}
    class_to_images = {cid: [] for cid in class_ids}
    for img_path in images:
        candidate_label = labels_dir / (img_path.stem + ".txt")
        if not candidate_label.exists():
            sibling = img_path.with_suffix(".txt")
            if sibling.exists():
                candidate_label = sibling
        boxes = read_yolo_labels(candidate_label)
        image_to_labels[img_path] = boxes
        present = set([b[0] for b in boxes])
        for cid in present:
            if cid in class_to_images:
                class_to_images[cid].append(img_path)

    out_root = Path(output_dir_str)
    out_root.mkdir(parents=True, exist_ok=True)
    num_sets = 3
    random.seed(seed)

    for set_idx in range(1, num_sets+1):
        set_dir = out_root / f"set_{set_idx}"
        for cid in class_ids:
            cname = class_names[cid]
            target_dir = set_dir / cname
            target_dir.mkdir(parents=True, exist_ok=True)
            available = class_to_images.get(cid, [])
            if not available:
                print(f"Warning: no images for class {cid} ({cname})")
                continue
            rng = random.Random(seed + cid)
            shuffled = available.copy()
            rng.shuffle(shuffled)
            groups = [[] for _ in range(num_sets)]
            for i,p in enumerate(shuffled):
                groups[i % num_sets].append(p)
            if groups[set_idx-1]:
                chosen_img = groups[set_idx-1][0]
                augment_mode = 0
            else:
                chosen_img = rng.choice(available)
                augment_mode = (set_idx-1) % 3
            img = cv2.imread(str(chosen_img))
            if img is None:
                print(f"Failed to read {chosen_img}, skipping.")
                continue
            boxes = image_to_labels.get(chosen_img, [])
            cls_boxes = [b for b in boxes if b[0] == cid]
            if not cls_boxes:
                print(f"Skipping {chosen_img.name}: no boxes of class {cid}")
                continue
            img_aug = augment_image(img, augment_mode)
            h,w = img.shape[:2]
            boxes_xyxy = []
            for b in cls_boxes:
                clsid, x1,y1,x2,y2 = yolo_to_xyxy(b, w, h)
                if augment_mode == 1:
                    x1f = w-1 - x2
                    x2f = w-1 - x1
                    x1, x2 = x1f, x2f
                boxes_xyxy.append((clsid, x1, y1, x2, y2))
            out_img = draw_boxes(img_aug, boxes_xyxy, cname)
            out_path = target_dir / f"{chosen_img.stem}_cls{cid}_set{set_idx}.jpg"
            cv2.imwrite(str(out_path), out_img)
            print(f"Saved {out_path}")
    print("Done.")

def main():
    parser = argparse.ArgumentParser(description="Visualize one-class-per-image sets from YOLO labels (improved)")
    parser.add_argument("--data-root", default="./train", help="Path to train folder or parent containing images/ and labels/")
    parser.add_argument("--output-dir", default="outputs", help="Where to save outputs")
    parser.add_argument("--names-file", default=None, help="Optional YAML file with names mapping")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    try:
        run_visualization(args.data_root, args.output_dir, args.names_file, args.seed)
    except FileNotFoundError as e:
        print(f"ERROR: {e}")
        print("Make sure your data root exists and contains images and label .txt files.")
        print("Example structures supported:")
        print("  train/images/*.jpg  +  train/labels/*.txt")
        print("  train/*.jpg  +  train/*.txt (co-located)")
        sys.exit(1)
    except Exception as e:
        print("An unexpected error occurred — full traceback below:")
        traceback.print_exc()
        sys.exit(1)

if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] [--data-root DATA_ROOT]
                             [--output-dir OUTPUT_DIR]
                             [--names-file NAMES_FILE] [--seed SEED]
ipykernel_launcher.py: error: unrecognized arguments: -f /home/pardhasarb/.local/share/jupyter/runtime/kernel-9d281f75-5ca3-44b6-8f65-5bb71ac83f2c.json


SystemExit: 2